# Causal Fine-Tuning of Multimodal Product Embeddings

**Note:** This notebook requires a GPU. In Colab, select *Runtime → Change runtime type → T4 GPU* (or better). On a T4, training takes about 100 minutes. If Colab disconnects, run the notebook again: training resumes after the last completed epoch.

In this lab, we map the raw text and the image data from Bach, et al. (2024) into raw *embeddings*. We train a multimodal neural network that combines RoBERTa for the text, BEiT for the image and SAINT for the tabular variables. The embeddings are saved separately.

For now, we use pre-trained encoders rather than fine-tuning. In the original paper, the network has two prediction heads, one for the quantity signal and one for the price signal, and minimizes

$$
L = \bigl\|Q_{it} - \hat{\ell}(X^{in}_i)\bigr\|_{\text{pred},2} \cdot
      \bigl\|P_{it} - \hat{m}(X^{in}_i)\bigr\|_{\text{pred},2},
$$

the product of the two root-mean-squared prediction errors.  The description and the image do not change over time, but part of the tabular input does: the offer counts, the lightning-deal indicator and the Buy Box fulfillment indicator are observed in every period. Following the paper's pipeline, we therefore compute an embedding for every product-period. The text and image parts, 1,536 of the 2,048 dimensions, are identical across the periods of a product, so most of the embedding cannot explain *changes* in prices.

## Installation

First, we install the required packages. `doubleml-deep` imports its SAINT tabular model when the package is imported, so `pytorch-widedeep` has to be importable even though we barely use it. A regular installation of `pytorch-widedeep` pulls in `spacy`, `gensim` and `opencv` and pins `numpy<2`, which downgrades numpy underneath Colab's pre-installed torch, pandas and scipy and breaks the runtime.

Instead, we install it with `--no-deps` and provide two small stub modules for the text utilities that `pytorch-widedeep` imports but that we never call. All other dependencies (`torchvision`, `cv2`, `wrapt`) are already available on Colab.

In [ ]:
%pip install -q lightning torchmetrics einops doubleml
%pip install -q --no-deps pytorch-widedeep
%pip install -q git+https://github.com/DoubleML/doubleml-deep.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


The following cell creates the two stub modules and adds them to the Python path.

In [ ]:
import os
import sys
import pathlib

# --- shim: satisfy pytorch-widedeep's unused text-utility imports -----------
# pytorch_widedeep.utils.text_utils imports gensim and spacy at module level.
# We never tokenise text with them (RoBERTa does that), so tiny stubs suffice
# and we avoid the numpy<2 downgrade their real versions would force.
SHIM = "/content/_wd_shim" if os.path.isdir("/content") else "_wd_shim"
for pkg in ("gensim", "spacy"):
    pathlib.Path(f"{SHIM}/{pkg}").mkdir(parents=True, exist_ok=True)

pathlib.Path(f"{SHIM}/gensim/__init__.py").write_text("")
pathlib.Path(f"{SHIM}/gensim/utils.py").write_text(
    "def tokenize(*a, **k):\n    raise NotImplementedError('gensim stub')\n")
pathlib.Path(f"{SHIM}/spacy/__init__.py").write_text(
    "__version__ = '0.0.0-stub'\n\n\n"
    "class Language:  # datasets.utils._dill checks issubclass(obj, spacy.Language)\n"
    "    pass\n\n\n"
    "def blank(*a, **k):\n    raise NotImplementedError('spacy stub')\n\n\n"
    "def load(*a, **k):\n    raise NotImplementedError('spacy stub')\n")
pathlib.Path(f"{SHIM}/spacy/symbols.py").write_text("ORTH = 65\n")

if SHIM not in sys.path:
    sys.path.insert(0, SHIM)

In [ ]:
import warnings

# Suppress DeprecationWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

Next, we import the required modules and check that a GPU is available.

In [ ]:
import gc
import glob
import io
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.random_projection import GaussianRandomProjection

from doubleml_deep import DoubleMLLightning
from doubleml_deep.plm import DoubleMLPLRDeep
from doubleml_deep.default_models import (
    DoubleMLDeepBeitModel,
    DoubleMLDeepRobertaModel,
    DoubleMLDeepSaintModel,
)

palette = sns.color_palette("colorblind")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.0f} GB)")
else:
    raise RuntimeError(
        "No GPU visible, so this notebook would train on CPU and take hours.\n"
        "Go to Runtime, Change runtime type, and pick a GPU (T4 is enough).\n"
        "TPU runtimes do not work here: torch sees no CUDA device and silently\n"
        "falls back to CPU. Supporting TPUs would need torch-xla.\n"
        "To override anyway, set ALLOW_CPU = True."
    )

GPU: Tesla T4 (16 GB)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


We mount Google Drive at the beginning rather than at the end (Colab only). Embedding outputs are written directly to the embeddings folder (`data/embeddings` locally, or `MyDrive/demand_labs_v2/embeddings` on Drive). Downloaded panel/image inputs are cached under `data/raw`.


In [ ]:
from pathlib import Path

# Use Google Drive in Colab; use persistent local/project storage elsewhere.
# Layout under the project root (main/ when run from main/code):
#   data/raw/         downloaded panel + image parquets (gitignored)
#   data/embeddings/  embedding outputs written directly here (tracked)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content")
    DATA_CACHE = PROJECT_DIR / "data" / "raw"
    OUTPUT_ROOT = Path("/content/drive/MyDrive/demand_labs_v2/embeddings")
    print("Using Google Drive for outputs.")
except ImportError:
    current_dir = Path.cwd()
    default_root = current_dir.parent if current_dir.name == "code" else current_dir
    PROJECT_DIR = Path(os.environ.get("DEMAND_REPRO_ROOT", default_root))
    DATA_CACHE = Path(os.environ.get("DEMAND_DATA_CACHE", PROJECT_DIR / "data" / "raw"))
    OUTPUT_ROOT = Path(os.environ.get("EMBEDDING_OUTPUT_DIR", PROJECT_DIR / "data" / "embeddings"))
    print(f"Using local/FASRC storage under {PROJECT_DIR}.")

DATA_CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = str(OUTPUT_ROOT)
print(f"data cache (raw): {DATA_CACHE}")
print(f"outputs (embeddings): {OUTPUT_ROOT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoints and artifact go to /content/drive/MyDrive/demand_labs_v2


## Constructing the Subsample

We construct the same 1,540 products as in Lab 1, using the same configuration and the same deterministic draw. If you change `N_PRODUCTS` or the list of image files in Lab 1, change them here as well.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration. Keep this cell identical in Labs 1 and 2.
# ---------------------------------------------------------------------------
BASE = ("https://raw.githubusercontent.com/JanTeichertKluge/"
        "demand-analysis-repro/main/main/data")

N_PRODUCTS = 1540  # products in the subsample
N_FINETUNE = 770   # of those, the ones Lab 2 trains on (the paper's I_1)
SEED = 42

panel_files = [
    "amzn_toys_monthly_diffs_ffill_fixed_splits_buybox/train-00000-of-00002.parquet",
    "amzn_toys_monthly_diffs_ffill_fixed_splits_buybox/train-00001-of-00002.parquet",
    "amzn_toys_monthly_diffs_ffill_fixed_splits_buybox/validation-00000-of-00002.parquet",
    "amzn_toys_monthly_diffs_ffill_fixed_splits_buybox/validation-00001-of-00002.parquet",
]

# An image file holds a fixed set of products and cannot be queried for a single
# product, so these five files determine the pool we draw from: 3,300 images, of
# which 1,540 belong to a product with a complete panel. Add more parts from the
# same folder for a larger sample.
image_files = [
    "amzn_toys_monthly_avg_long_ffill_28_7_images/train-00000-of-00004-part000.parquet",
    "amzn_toys_monthly_avg_long_ffill_28_7_images/train-00001-of-00004-part000.parquet",
    "amzn_toys_monthly_avg_long_ffill_28_7_images/train-00002-of-00004-part000.parquet",
    "amzn_toys_monthly_avg_long_ffill_28_7_images/train-00003-of-00004-part000.parquet",
    "amzn_toys_monthly_avg_long_ffill_28_7_images/train-00000-of-00004-part001.parquet",
]

for remote in panel_files + image_files:
    local = DATA_CACHE / remote.split("/")[-1]
    if not local.exists():
        print("downloading", local, flush=True)
        urllib.request.urlretrieve(f"{BASE}/{remote}", local)

print(f"data ready in {DATA_CACHE}")

data ready (~500 MB, cached for this session)


Next, we load the panel and draw the subsample exactly as in Lab 1.

In [ ]:
columns = ["ASIN", "window", "date", "SALES_RANK", "BUYBOX_PRICE", "text",
           "RATING", "REVIEW_COUNT", "subcat_aggregated",
           "New Offer Count: Current",
           "Count of retrieved live offers: New, FBA",
           "Count of retrieved live offers: New, FBM",
           "Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]

panel = pd.concat([pd.read_parquet(DATA_CACHE / f.split("/")[-1], columns=columns)
                   for f in panel_files], ignore_index=True)

# Keep the 28-day time averages, then every fourth week: 14 four-week periods
# from 2023-01-30 to 2024-01-29. SALES_RANK is already log(1 / sales rank) and
# BUYBOX_PRICE already log(price), so they are the paper's Q_it and P_it.
panel = panel[panel["window"] == 28].drop(columns="window")
dates = np.sort(panel["date"].unique())
panel["date_t"] = panel["date"].map({d: i for i, d in enumerate(dates)})
panel = panel[(panel["date_t"] <= 54) & (panel["date_t"] % 4 == 0)].dropna()
panel["period"] = panel["date_t"] // 4
panel = panel.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})

# Eligible products: we have an image, and they are observed in all 14 periods.
have_image = set(pd.concat(
    [pd.read_parquet(DATA_CACHE / f.split("/")[-1], columns=["ASIN"])
     for f in image_files], ignore_index=True)["ASIN"])
n_periods = panel.groupby("ASIN")["period"].nunique()
candidates = np.array(sorted(set(n_periods[n_periods == 14].index) & have_image))

# Deterministic draw, then a split by product. Labs 4 and 5 estimate only on
# the "estimation" products, so the embeddings they use were fitted on different
# products. The paper's Addendum A relies on that independence.
rng = np.random.default_rng(SEED)
chosen = np.sort(rng.choice(candidates, size=N_PRODUCTS, replace=False))
finetune = set(chosen[rng.permutation(N_PRODUCTS)[:N_FINETUNE]])

df = panel[panel["ASIN"].isin(set(chosen))].sort_values(["ASIN", "period"])
df = df.reset_index(drop=True)
df["split"] = np.where(df["ASIN"].isin(finetune), "finetune", "estimation")

# Lagged state and first differences, within product
grouped = df.groupby("ASIN")
df["Q_t-1"] = grouped["Q_t"].shift(1)
df["P_bb_t-1"] = grouped["P_bb_t"].shift(1)
df["RATING_t-1"] = grouped["RATING"].shift(1)
df["REVIEW_COUNT_t-1"] = grouped["REVIEW_COUNT"].shift(1)
df["Delta_Q_t"] = df["Q_t"] - df["Q_t-1"]
df["Delta_P_bb_t"] = df["P_bb_t"] - df["P_bb_t-1"]

print(f"{len(candidates):,} products qualify; using {N_PRODUCTS}")
print(df.groupby("split").agg(products=("ASIN", "nunique"), rows=("ASIN", "size")))

1,540 products qualify; using 1540
            products   rows
split                      
estimation       770  10780
finetune         770  10780


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Training Settings

We use the settings of the paper's training script (`_src_scrpts/training_level_v02_bert_hpc.py`, the time-independent text-and-image run). The cell also fixes the session budget, the validation periods, and the dimensions used for the dimension reduction below.

In [ ]:
torch.set_float32_matmul_precision("medium")

BATCH_SIZE = 16
ACCUMULATE_GRAD_BATCHES = 2   # optimizer step every 2 batches: effective batch of 32
LEARNING_RATE = 2e-5
MOMENTUM = 0.95               # passed through as in the script; AdamW ignores it
WD = 1e-6
SCHEDULER = None              # constant learning rate
GRADIENT_CLIP_VAL = 2
DROPOUT = 0.1
FUSION_HEAD = [256, 128]

MAX_EPOCHS = 10               # upper bound; early stopping ends the run sooner
PATIENCE = 2                  # epochs without improvement before stopping
MIN_DELTA = 0.001             # smallest drop in val_combined_loss that counts

# Colab: how long this session may train (the rest of the notebook needs about
# 30 minutes more), and whether to continue a run an earlier session started.
TRAIN_HOURS = 2
RESUME = True

# Validation decides when training stops and which epoch's weights we keep:
# every estimation product, observed in two periods.
VAL_PERIODS = [0, 7]

N_CLUSTERS = 5          # k in k-means, as in the paper
N_COMPONENTS = 5        # number of PCA / similarity features
PROJECTION_DIM = 256    # Johnson-Lindenstrauss target dimension

torch.manual_seed(SEED)
pl.seed_everything(SEED, workers=True)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


42

Next, we load the product images. The source images are large: half of them are at least 1,500 pixels wide, and the largest are 2,560 × 2,560 pixels, which amounts to 19 MB once decoded. We therefore keep them encoded (about 130 KB each on average) and decode them one at a time during preprocessing below, so that the image processor sees exactly the images that the paper's pipeline saw.

In [ ]:
images = {}
for f in image_files:
    table = pd.read_parquet(DATA_CACHE / f.split("/")[-1])
    hit = table[table["ASIN"].isin(set(chosen))]
    images.update(zip(hit["ASIN"], (record["bytes"] for record in hit["image"])))
    del table, hit

megabytes = sum(len(b) for b in images.values()) / 1e6
print(f"{len(images):,} product images ({megabytes:.0f} MB, still encoded)")

1,540 product images (203 MB, still encoded)


## Assembling the Inputs

The tabular input follows the paper's training script: the three offer counts (standardized), the lightning-deal indicator, the Buy Box FBA indicator and the subcategory dummies. Ratings, review counts, and lagged prices and quantities are not fed into the network; they enter later as ordinary controls in Labs 3 to 5.

The data are split by product. The network is trained on all periods of the fine-tuning products, so it sees each product 14 times with the same text and image but with different targets and tabular values. For validation, we use all estimation products in two periods. The validation loss determines when training stops and which epoch's weights we keep, just as the paper's script validated on held-out products. The network is never trained on these rows.

In [ ]:
# Fixed category list, so the tabular input width does not depend on which
# products happen to be drawn.
SUBCATS = sorted(panel["subcat_aggregated"].unique())
SUBCAT_COLS = [f"sub_{c.replace(' ', '_').replace('&', 'and')}" for c in SUBCATS]

dummies = (pd.get_dummies(df["subcat_aggregated"])
           .reindex(columns=SUBCATS, fill_value=0)
           .astype("float32"))
dummies.columns = SUBCAT_COLS
df = pd.concat([df, dummies], axis=1)

# Network copies of the time-varying tabular inputs; the artifact keeps the raw
# values. The script fits one scaler per split. We fit on the fine-tune rows and
# apply it to every row, so the estimation products do not shape the scaling.
COUNT_COLS = ["New Offer Count: Current",
              "Count of retrieved live offers: New, FBA",
              "Count of retrieved live offers: New, FBM"]
FLAG_COLS = ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
X_COUNT_COLS = ["x_n_offers", "x_n_offers_fba", "x_n_offers_fbm"]
X_FLAG_COLS = ["x_lightning_deal", "x_is_fba"]

scaler = StandardScaler().fit(df.loc[df["split"] == "finetune", COUNT_COLS])
df[X_COUNT_COLS] = scaler.transform(df[COUNT_COLS])
df[X_FLAG_COLS] = df[FLAG_COLS].astype("float32").to_numpy()

TAB_NET_COLS = X_COUNT_COLS + X_FLAG_COLS + SUBCAT_COLS
print(f"{len(TAB_NET_COLS)} tabular columns ({len(SUBCAT_COLS)} subcategory dummies) "
      f"-> SAINT embedding of {32 * len(TAB_NET_COLS)} dimensions")

# Row positions in df: every period of the fine-tune products for training,
# every estimation product in VAL_PERIODS for validation.
is_finetune = (df["split"] == "finetune").to_numpy()
train_rows = np.flatnonzero(is_finetune)
val_rows = np.flatnonzero(~is_finetune & df["period"].isin(VAL_PERIODS).to_numpy())
print(f"training rows:   {len(train_rows):,} ({df.loc[train_rows, 'ASIN'].nunique()} products)")
print(f"validation rows: {len(val_rows):,} ({df.loc[val_rows, 'ASIN'].nunique()} products)")

16 tabular columns (11 subcategory dummies) -> SAINT embedding of 512 dimensions
training rows:   10,780 (770 products)
validation rows: 1,540 (770 products)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


The tabular input has 16 columns, 11 of which are subcategory dummies, and SAINT maps them into a 512-dimensional embedding. The network is trained on the 10,780 rows of the 770 fine-tuning products and validated on 1,540 rows of the 770 estimation products.

## Pretrained embedding extraction

We load pretrained text and image encoders and run a forward pass only (no quantity/price fine-tuning). Tabular inputs stay as the assembled feature columns. Raw modality arrays are saved before any random projection or PCA.


In [ ]:
import copy
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoImageProcessor,
    AutoModel,
    AutoTokenizer,
)


TEXT_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"
IMAGE_ID = "microsoft/beit-base-patch16-224-pt22k-ft22k"

BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_EPOCHS = 10
TABULAR_EMBEDDING_DIM = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Write embedding arrays directly into data/embeddings (OUTPUT_ROOT).
OUTPUT_DIR = Path(DRIVE_DIR or ".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"device: {DEVICE}")
print(f"outputs: {OUTPUT_DIR}")


In [ ]:
class TextQuantityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(TEXT_ID)
        self.head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, batch):
        text_embedding = self.encoder(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).last_hidden_state[:, 0, :]

        prediction = self.head(text_embedding).squeeze(1)
        return prediction, text_embedding


class TabularQuantityModel(nn.Module):
    def __init__(self, n_columns):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_columns, TABULAR_EMBEDDING_DIM),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(TABULAR_EMBEDDING_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, batch):
        tabular_embedding = self.encoder(batch["tabular"])
        prediction = self.head(tabular_embedding).squeeze(1)
        return prediction, tabular_embedding


class ImageQuantityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(IMAGE_ID)
        self.head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, batch):
        image_embedding = self.encoder(
            pixel_values=batch["pixel_values"],
        ).last_hidden_state[:, 0, :]

        prediction = self.head(image_embedding).squeeze(1)
        return prediction, image_embedding


In [ ]:
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_ID)
image_processor = AutoImageProcessor.from_pretrained(IMAGE_ID)


class TextDataset(Dataset):
    def __init__(self, df, row_ids):
        self.df = df
        self.row_ids = np.asarray(row_ids)

    def __len__(self):
        return len(self.row_ids)

    def __getitem__(self, i):
        row = self.df.iloc[self.row_ids[i]]
        return {"text": str(row["text"]), "target": float(row["Q_t"])}


def collate_text(batch):
    tokens = text_tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    tokens["target"] = torch.tensor(
        [row["target"] for row in batch], dtype=torch.float32
    )
    return tokens


class TabularDataset(Dataset):
    def __init__(self, df, row_ids):
        self.df = df
        self.row_ids = np.asarray(row_ids)

    def __len__(self):
        return len(self.row_ids)

    def __getitem__(self, i):
        row = self.df.iloc[self.row_ids[i]]
        return {
            "tabular": torch.tensor(
                row[TAB_NET_COLS].to_numpy(dtype="float32")
            ),
            "target": torch.tensor(float(row["Q_t"])),
        }


class ImageDataset(Dataset):
    def __init__(self, df, row_ids):
        self.df = df
        self.row_ids = np.asarray(row_ids)

    def __len__(self):
        return len(self.row_ids)

    def __getitem__(self, i):
        row = self.df.iloc[self.row_ids[i]]
        image = Image.open(io.BytesIO(images[row["ASIN"]])).convert("RGB")
        pixels = image_processor(images=image, return_tensors="pt")["pixel_values"][0]
        return {
            "pixel_values": pixels,
            "target": torch.tensor(float(row["Q_t"])),
        }


In [ ]:
# Train/val loaders omitted: this notebook extracts pretrained embeddings only.


In [ ]:
def move_batch_to_device(batch):
    return {
        key: value.to(DEVICE) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


# Quantity/price fine-tuning disabled — pretrained forward pass only.


In [ ]:
# Pretrained encoders only (eval). No quantity/price training.
text_model = TextQuantityModel().to(DEVICE).eval()
image_model = ImageQuantityModel().to(DEVICE).eval()
print("loaded pretrained text and image encoders (forward pass only)")


In [ ]:
def extract_embeddings(
    texts,
    image_bytes_list,
    text_model,
    image_model,
    output_dir=OUTPUT_DIR,
    tabular_E=None,
    embedding_rows=None,
    batch_size=BATCH_SIZE,
):
    """Extract text/image embeddings from raw inputs and save them.

    Row i of every returned array aligns with texts[i] / image_bytes_list[i]
    (and with embedding_rows.iloc[i] when provided), matching the Bach et al.
    product-period ordering convention.
    """
    text_model.eval()
    image_model.eval()
    text_chunks, image_chunks = [], []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            end = start + batch_size
            tokens = text_tokenizer(
                list(texts[start:end]),
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            text_batch = {
                "input_ids": tokens["input_ids"].to(DEVICE),
                "attention_mask": tokens["attention_mask"].to(DEVICE),
            }
            _, text_embedding = text_model(text_batch)
            text_chunks.append(text_embedding.cpu().numpy())

            pixels = []
            for raw in image_bytes_list[start:end]:
                image = Image.open(io.BytesIO(raw)).convert("RGB")
                pixels.append(
                    image_processor(images=image, return_tensors="pt")["pixel_values"][0]
                )
            image_batch = {"pixel_values": torch.stack(pixels).to(DEVICE)}
            _, image_embedding = image_model(image_batch)
            image_chunks.append(image_embedding.cpu().numpy())

    text_E = np.concatenate(text_chunks, axis=0)
    image_E = np.concatenate(image_chunks, axis=0)
    parts = [text_E]
    if tabular_E is not None:
        parts.append(np.asarray(tabular_E))
    parts.append(image_E)
    E = np.concatenate(parts, axis=1)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    np.save(output_dir / "text_embeddings.npy", text_E, allow_pickle=False)
    np.save(output_dir / "image_embeddings.npy", image_E, allow_pickle=False)
    np.save(output_dir / "raw_embeddings.npy", E, allow_pickle=False)
    if tabular_E is not None:
        np.save(output_dir / "tabular_embeddings.npy", tabular_E, allow_pickle=False)

    if embedding_rows is not None:
        meta = embedding_rows.reset_index(drop=True)

        def save_embedding_parquet(embedding, prefix, filename):
            columns = [f"{prefix}_{i:04d}" for i in range(embedding.shape[1])]
            table = pd.concat(
                [meta, pd.DataFrame(embedding, columns=columns)],
                axis=1,
            )
            table.to_parquet(output_dir / filename, index=False)

        save_embedding_parquet(text_E, "text", "text_embeddings.parquet")
        if tabular_E is not None:
            save_embedding_parquet(tabular_E, "tabular", "tabular_embeddings.parquet")
        save_embedding_parquet(image_E, "image", "image_embeddings.parquet")
        save_embedding_parquet(E, "combined", "raw_embeddings.parquet")
        meta.to_csv(output_dir / "embedding_rows.csv", index=False)

    print(f"saved NumPy and Parquet embeddings to {output_dir}")
    return text_E, image_E, E


# Bach-style ASIN -> encoded image bytes (reload if earlier cell was skipped).
if "images" not in globals() or not images:
    images = {}
    for f in image_files:
        table = pd.read_parquet(DATA_CACHE / f.split("/")[-1])
        hit = table[table["ASIN"].isin(set(chosen))]
        images.update(zip(hit["ASIN"], (record["bytes"] for record in hit["image"])))
        del table, hit
    print(f"loaded {len(images):,} product images for extraction")

# Raw text + image bytes in df order (one row per product-period).
texts = df["text"].astype(str).tolist()
image_bytes_list = [images[asin] for asin in df["ASIN"]]
embedding_rows = df[["ASIN", "date", "period", "split", "Q_t"]].reset_index(drop=True)
# Structured tabular columns (no learned tabular encoder / no Q-P training).
tabular_E = df[TAB_NET_COLS].to_numpy(dtype="float32")

text_E, image_E, E = extract_embeddings(
    texts,
    image_bytes_list,
    text_model,
    image_model,
    tabular_E=tabular_E,
    embedding_rows=embedding_rows,
)

assert len(text_E) == len(df), "embedding rows are not aligned with df"
print("text embeddings:", text_E.shape)
print("tabular embeddings:", tabular_E.shape)
print("image embeddings:", image_E.shape)
print("combined raw embeddings:", E.shape)


## Saving raw embeddings

`extract_embeddings` writes these arrays before random projection, PCA, or clustering. The separate arrays preserve the modality-specific representations; `raw_embeddings.npy` is their concatenation for later projection. Row order matches `df` (ASIN / date / period / split), as in Bach et al.


In [ ]:
# Files written by extract_embeddings directly into OUTPUT_DIR (data/embeddings):
#   text_embeddings.npy / .parquet
#   tabular_embeddings.npy / .parquet
#   image_embeddings.npy / .parquet
#   raw_embeddings.npy / .parquet
#   embedding_rows.csv
print("outputs:", sorted(p.name for p in OUTPUT_DIR.glob("*embeddings*")))
print("row index:", OUTPUT_DIR / "embedding_rows.csv")
